# Controlled AI-Document Generation Feasibility Study (Colab GPU Launcher)

> **Tên đề tài chính thức**: “Nghiên cứu và xây dựng công cụ phát hiện ảnh và tài liệu giả mạo sinh bởi AI.”<br>
> **Mục tiêu**: Thử nghiệm khả thi tạo dữ liệu trên đúng **bốn tài liệu CORD v2** (2 replacement, 2 removal) bằng `diffusers/stable-diffusion-xl-1.0-inpainting-0.1`.<br>
> **Ranh giới**: Đây là generation feasibility study, KHÔNG phải detector training, KHÔNG phải cohort đánh giá độc lập.<br>
> **Không yêu cầu token/đăng nhập**: Chạy hoàn toàn bằng tài khoản Google Colab miễn phí thông thường (GPU T4/V100/A100). Không yêu cầu Hugging Face token.

### Quy Trình 4 Bước Thực Hiện Trên Colab:
1. **Cell 1**: Kiểm tra phần cứng GPU và cài đặt thư viện cần thiết (`diffusers`, `transformers`, `accelerate`, `safetensors`, `pyarrow`).
2. **Cell 2**: Nạp kế hoạch tiền đăng ký `document_feasibility_plan.json` và mã nguồn thực thi `run_document_feasibility_colab.py`.
3. **Cell 3**: Thực thi 4 attempts đã khóa cứng: tải checkpoint FP16 (6.46 GiB), nạp 4 ảnh CORD-v2, inpainting suy luận, composite và kiểm tra bitwise equality ngoài mask.
4. **Cell 4**: Tải gói kết quả `document_feasibility_results.zip` về máy LOCAL để kiểm chứng kỹ thuật và thẩm định nội dung.


In [ ]:
# Cell 1: Kiểm tra phần cứng GPU và cài đặt dependencies
!nvidia-smi

print("Installing dependencies (diffusers, transformers, accelerate, safetensors, pyarrow)...")
!pip install -q "diffusers>=0.21.0" "transformers>=4.34.0" "accelerate>=0.23.0" safetensors pyarrow pillow

import torch
assert torch.cuda.is_available(), "LỖI: Chưa kích hoạt GPU! Vui lòng vào Runtime -> Change runtime type -> Chọn T4 GPU!"
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"✓ GPU sẵn sàng: {gpu_name} ({vram_gb:.2f} GB VRAM)")
assert vram_gb >= 12.0, f"CẢNH BÁO: VRAM ({vram_gb:.2f} GB) có thể không đủ cho SDXL FP16 (khuyến nghị >= 12 GB)!"


In [ ]:
# Cell 2: Staging kế hoạch máy tiền đăng ký và runner script
import json
from pathlib import Path

PLAN_DATA = {
  "metadata": {
    "study_name": "Controlled AI-Document Generation Feasibility Study",
    "protocol_version": "1.1.0",
    "official_project_title": "Nghi\u00ean c\u1ee9u v\u00e0 x\u00e2y d\u1ef1ng c\u00f4ng c\u1ee5 ph\u00e1t hi\u1ec7n \u1ea3nh v\u00e0 t\u00e0i li\u1ec7u gi\u1ea3 m\u1ea1o sinh b\u1edfi AI.",
    "execution_target": "GOOGLE_COLAB_GPU_EPHEMERAL",
    "local_role": "PLAN_PREPARATION_AND_INDEPENDENT_VERIFICATION",
    "dataset": {
      "repository": "naver-clova-ix/cord-v2",
      "split": "validation",
      "revision": "7f0115a4b758a71d6473b8d085751692da2fef98",
      "license": "CC-BY 4.0",
      "parquet_file": "data/validation-00000-of-00001-cc3c5779fe22e8ca.parquet"
    },
    "model": {
      "repository": "diffusers/stable-diffusion-xl-1.0-inpainting-0.1",
      "revision": "115134f363124c53c7d878647567d04daf26e41e",
      "license": "OpenRAIL++",
      "pipeline_class": "StableDiffusionXLInpaintPipeline",
      "torch_dtype": "float16",
      "variant": "fp16",
      "canvas_size": 1024,
      "scheduler": "EulerDiscreteScheduler",
      "num_inference_steps": 30,
      "guidance_scale": 7.5,
      "strength": 0.99
    },
    "quarantine_policy": "DEVELOPMENT_FEASIBILITY_EXCLUDED_FROM_EVALUATION",
    "budget_limits": {
      "checkpoint_colab_max_bytes": 12884901888,
      "original_data_colab_max_bytes": 209715200,
      "artifacts_transfer_max_bytes": 104857600,
      "max_gpu_minutes": 60
    }
  },
  "attempts": [
    {
      "attempt_index": 0,
      "attempt_id": "attempt_0_replacement_price",
      "source_row_index": 22,
      "source_image_id": 22,
      "source_doc_identifier": "cord_v2_valid_row22_img22",
      "native_size": {
        "width": 576,
        "height": 864
      },
      "operation": "replacement",
      "field_category": "menu.price",
      "original_text": "280,000",
      "target_text": "350,000",
      "target_bbox": [
        442,
        406,
        468,
        488
      ],
      "context_margin_ratio": 0.5,
      "prompt": "a photo of a retail receipt, crisp clear printed text displaying '350,000', matching authentic thermal receipt paper texture, sharp dark printed typography, clean alignment",
      "negative_prompt": "blurry, low quality, distorted text, illegible fonts, messy background, watermarks, artifacts, colored stains, handwritten",
      "seed": 20261012,
      "rationale": "Evaluate numerical glyph replacement on thermal paper texture without disturbing neighboring line items."
    },
    {
      "attempt_index": 1,
      "attempt_id": "attempt_1_replacement_item_name",
      "source_row_index": 61,
      "source_image_id": 61,
      "source_doc_identifier": "cord_v2_valid_row61_img61",
      "native_size": {
        "width": 1356,
        "height": 2315
      },
      "operation": "replacement",
      "field_category": "menu.nm",
      "original_text": "Populaire Chocolate",
      "target_text": "Magnum Gold Almond",
      "target_bbox": [
        1085,
        128,
        1156,
        749
      ],
      "context_margin_ratio": 0.4,
      "prompt": "a high resolution photo of a store receipt, sharp printed text displaying 'Magnum Gold Almond', matching receipt typography and texture, uniform lighting, crisp black font",
      "negative_prompt": "blurry, low quality, distorted text, illegible fonts, crooked baseline, stains, artifacts",
      "seed": 20261013,
      "rationale": "Evaluate long alphanumeric product name substitution on high-resolution receipt."
    },
    {
      "attempt_index": 2,
      "attempt_id": "attempt_2_removal_discount_line",
      "source_row_index": 7,
      "source_image_id": 7,
      "source_doc_identifier": "cord_v2_valid_row7_img7",
      "native_size": {
        "width": 576,
        "height": 864
      },
      "operation": "removal",
      "field_category": "sub_total.discount_price",
      "original_text": "Discount: -9,545",
      "target_text": "",
      "target_bbox": [
        524,
        208,
        556,
        492
      ],
      "context_margin_ratio": 0.5,
      "prompt": "a photo of a retail receipt, clean blank receipt paper texture, subtle paper grain, uniform off-white surface, no text, completely erased",
      "negative_prompt": "text, numbers, letters, blurry stains, dark spots, ghosting, visible outlines, artifacts",
      "seed": 20261014,
      "rationale": "Evaluate complete semantic erasure of discount subtotal line with seamless paper background restoration."
    },
    {
      "attempt_index": 3,
      "attempt_id": "attempt_3_removal_tax_line",
      "source_row_index": 93,
      "source_image_id": 93,
      "source_doc_identifier": "cord_v2_valid_row93_img93",
      "native_size": {
        "width": 864,
        "height": 1296
      },
      "operation": "removal",
      "field_category": "sub_total.tax_price",
      "original_text": "TAX 15,636",
      "target_text": "",
      "target_bbox": [
        939,
        383,
        975,
        605
      ],
      "context_margin_ratio": 0.5,
      "prompt": "a photo of a retail receipt, clean blank receipt paper texture, subtle vertical paper fiber, uniform flat lighting, seamless empty paper, no text",
      "negative_prompt": "text, numbers, letters, ghost glyphs, smudges, stains, artifacts, border lines",
      "seed": 20261015,
      "rationale": "Evaluate erasure of tax line between subtotal and grand total while preserving surrounding table layout."
    }
  ]
}

plan_file = Path('document_feasibility_plan.json')
with open(plan_file, 'w', encoding='utf-8') as f:
    json.dump(PLAN_DATA, f, indent=2, ensure_ascii=False)
print(f"✓ Đã ghi kế hoạch tiền đăng ký vào: {plan_file.resolve()}")
print(f"  - Nghiên cứu: {PLAN_DATA['metadata']['study_name']}")
  - Model: {PLAN_DATA['metadata']['model']['repository']} @ {PLAN_DATA['metadata']['model']['revision'][:8]}")
  - Attempts: {len(PLAN_DATA['attempts'])} ({[a['attempt_id'] for a in PLAN_DATA['attempts']]})")

# Staging runner script
with open('run_document_feasibility_colab.py', 'w', encoding='utf-8') as rf:
    rf.write("#!/usr/bin/env python3\n\"\"\"Runner script for Google Colab GPU execution of Document Generation Feasibility Study.\n\nPerforms preflight checks, FP16 safetensors checkpoint downloading without tokens,\nCORD-v2 source extraction, inpainting inference, inverse geometry compositing,\nreceipt generation, and packaging of document_feasibility_results.zip.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport io\nimport json\nimport os\nimport shutil\nimport socket\nimport sys\nimport time\nimport zipfile\nfrom pathlib import Path\nfrom typing import Any, Dict, List, Optional, Tuple\n\nimport numpy as np\nfrom PIL import Image\n\n# Apply Windows socket port exclusion safety patch if running on Windows\nif sys.platform == \"win32\":\n    _orig_create_conn = socket.create_connection\n\n    def _safe_create_connection(address, timeout=socket._GLOBAL_DEFAULT_TIMEOUT, source_address=None):\n        if source_address is None:\n            for p in range(51000, 55000):\n                try:\n                    return _orig_create_conn(address, timeout=timeout, source_address=(\"0.0.0.0\", p))\n                except (PermissionError, OSError):\n                    continue\n        return _orig_create_conn(address, timeout=timeout, source_address=source_address)\n\n    socket.create_connection = _safe_create_connection\n\n\n# Standalone Geometry & Composite functions for complete portability on Colab\ndef extract_roi_and_build_canvas(\n    image: Image.Image,\n    target_bbox: List[int] | Tuple[int, int, int, int],\n    canvas_size: int = 1024,\n    context_margin_ratio: float = 0.5,\n) -> Dict[str, Any]:\n    if image.mode != \"RGB\":\n        image = image.convert(\"RGB\")\n    orig_w, orig_h = image.size\n    ymin, xmin, ymax, xmax = [int(v) for v in target_bbox]\n    ymin = max(0, min(orig_h - 1, ymin))\n    ymax = max(ymin + 1, min(orig_h, ymax))\n    xmin = max(0, min(orig_w - 1, xmin))\n    xmax = max(xmin + 1, min(orig_w, xmax))\n\n    bbox_w = xmax - xmin\n    bbox_h = ymax - ymin\n    margin_x = int(round(bbox_w * context_margin_ratio))\n    margin_y = int(round(bbox_h * context_margin_ratio))\n\n    crop_ymin = max(0, ymin - margin_y)\n    crop_ymax = min(orig_h, ymax + margin_y)\n    crop_xmin = max(0, xmin - margin_x)\n    crop_xmax = min(orig_w, xmax + margin_x)\n\n    crop_w = crop_xmax - crop_xmin\n    crop_h = crop_ymax - crop_ymin\n\n    context_crop = image.crop((crop_xmin, crop_ymin, crop_xmax, crop_ymax))\n\n    target_rel_ymin = ymin - crop_ymin\n    target_rel_ymax = ymax - crop_ymin\n    target_rel_xmin = xmin - crop_xmin\n    target_rel_xmax = xmax - crop_xmin\n\n    mask_crop_arr = np.zeros((crop_h, crop_w), dtype=np.uint8)\n    mask_crop_arr[target_rel_ymin:target_rel_ymax, target_rel_xmin:target_rel_xmax] = 255\n    mask_crop = Image.fromarray(mask_crop_arr, mode=\"L\")\n\n    scale = min(canvas_size / crop_w, canvas_size / crop_h)\n    scaled_w = max(1, min(canvas_size, int(round(crop_w * scale))))\n    scaled_h = max(1, min(canvas_size, int(round(crop_h * scale))))\n\n    scaled_image = context_crop.resize((scaled_w, scaled_h), resample=Image.Resampling.BICUBIC)\n    scaled_mask = mask_crop.resize((scaled_w, scaled_h), resample=Image.Resampling.NEAREST)\n    scaled_mask_arr = (np.array(scaled_mask) > 127).astype(np.uint8) * 255\n    scaled_mask = Image.fromarray(scaled_mask_arr, mode=\"L\")\n\n    pad_x = (canvas_size - scaled_w) // 2\n    pad_y = (canvas_size - scaled_h) // 2\n\n    canvas_image = Image.new(\"RGB\", (canvas_size, canvas_size), color=(128, 128, 128))\n    canvas_image.paste(scaled_image, (pad_x, pad_y))\n\n    canvas_mask = Image.new(\"L\", (canvas_size, canvas_size), color=0)\n    canvas_mask.paste(scaled_mask, (pad_x, pad_y))\n\n    receipt = {\n        \"original_size\": [orig_w, orig_h],\n        \"target_bbox\": [ymin, xmin, ymax, xmax],\n        \"crop_bbox\": [crop_ymin, crop_xmin, crop_ymax, crop_xmax],\n        \"crop_size\": [crop_w, crop_h],\n        \"scale\": float(scale),\n        \"scaled_size\": [scaled_w, scaled_h],\n        \"padding\": [pad_y, pad_x],\n        \"canvas_size\": canvas_size,\n        \"context_margin_ratio\": float(context_margin_ratio),\n    }\n\n    return {\n        \"canvas_image\": canvas_image,\n        \"canvas_mask\": canvas_mask,\n        \"context_crop\": context_crop,\n        \"geometry_receipt\": receipt,\n    }\n\n\ndef composite_inpainted_patch(\n    original_image: Image.Image,\n    model_output_canvas: Image.Image,\n    geometry_receipt: Dict[str, Any],\n) -> Dict[str, Any]:\n    if original_image.mode != \"RGB\":\n        original_image = original_image.convert(\"RGB\")\n    if model_output_canvas.mode != \"RGB\":\n        model_output_canvas = model_output_canvas.convert(\"RGB\")\n\n    orig_w, orig_h = geometry_receipt[\"original_size\"]\n    crop_ymin, crop_xmin, crop_ymax, crop_xmax = geometry_receipt[\"crop_bbox\"]\n    crop_w, crop_h = geometry_receipt[\"crop_size\"]\n    scaled_w, scaled_h = geometry_receipt[\"scaled_size\"]\n    pad_y, pad_x = geometry_receipt[\"padding\"]\n    target_ymin, target_xmin, target_ymax, target_xmax = geometry_receipt[\"target_bbox\"]\n\n    unpadded_patch = model_output_canvas.crop((pad_x, pad_y, pad_x + scaled_w, pad_y + scaled_h))\n    restored_patch = unpadded_patch.resize((crop_w, crop_h), resample=Image.Resampling.BICUBIC)\n    restored_patch_arr = np.array(restored_patch)\n\n    orig_arr = np.array(original_image)\n    candidate_full_arr = orig_arr.copy()\n    candidate_full_arr[crop_ymin:crop_ymax, crop_xmin:crop_xmax] = restored_patch_arr\n\n    final_mask_arr = np.zeros((orig_h, orig_w), dtype=np.uint8)\n    final_mask_arr[target_ymin:target_ymax, target_xmin:target_xmax] = 255\n\n    final_edited_arr = np.where(final_mask_arr[:, :, None] == 255, candidate_full_arr, orig_arr)\n\n    return {\n        \"final_edited_image\": Image.fromarray(final_edited_arr, mode=\"RGB\"),\n        \"final_intervention_mask\": Image.fromarray(final_mask_arr, mode=\"L\"),\n        \"composite_receipt\": {\n            \"status\": \"COMPOSITED\",\n            \"original_size\": [orig_w, orig_h],\n            \"crop_bbox\": [crop_ymin, crop_xmin, crop_ymax, crop_xmax],\n            \"target_bbox\": [target_ymin, target_xmin, target_ymax, target_xmax],\n        },\n    }\n\n\ndef verify_outside_equality(\n    authentic_image: Image.Image | np.ndarray,\n    edited_image: Image.Image | np.ndarray,\n    intervention_mask: Image.Image | np.ndarray,\n) -> Tuple[bool, int]:\n    auth_arr = np.array(authentic_image, dtype=np.uint8)\n    edit_arr = np.array(edited_image, dtype=np.uint8)\n    mask_arr = np.array(intervention_mask, dtype=np.uint8)\n    outside = mask_arr == 0\n    if not np.any(outside):\n        return True, 0\n    auth_out = auth_arr[outside]\n    edit_out = edit_arr[outside]\n    is_eq = bool(np.array_equal(auth_out, edit_out))\n    if is_eq:\n        return True, 0\n    max_d = int(np.max(np.abs(auth_out.astype(np.int32) - edit_out.astype(np.int32))))\n    return False, max_d\n\n\ndef compute_sha256(file_path: Path) -> str:\n    h = hashlib.sha256()\n    with open(file_path, \"rb\") as f:\n        while chunk := f.read(65536):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef fetch_cord_source_image(row_idx: int, parquet_url: str) -> Image.Image:\n    \"\"\"Fetch single source receipt image from CORD-v2 parquet row.\"\"\"\n    import urllib.request\n    import pyarrow.parquet as pq\n\n    class RangeFile(io.BufferedIOBase):\n        def __init__(self, url: str):\n            self.url = url\n            req = urllib.request.Request(url, method=\"HEAD\", headers={\"User-Agent\": \"Mozilla/5.0\"})\n            with urllib.request.urlopen(req) as resp:\n                self.total_size = int(resp.headers.get(\"Content-Length\", 0))\n            self.pos = 0\n\n        def readable(self): return True\n        def seekable(self): return True\n        def seek(self, offset, whence=io.SEEK_SET):\n            if whence == io.SEEK_SET: self.pos = offset\n            elif whence == io.SEEK_CUR: self.pos += offset\n            elif whence == io.SEEK_END: self.pos = self.total_size + offset\n            return self.pos\n        def tell(self): return self.pos\n        def read(self, size=-1):\n            if size == -1 or self.pos + size > self.total_size: size = self.total_size - self.pos\n            if size <= 0: return b\"\"\n            req = urllib.request.Request(self.url, headers={\"Range\": f\"bytes={self.pos}-{self.pos + size - 1}\", \"User-Agent\": \"Mozilla/5.0\"})\n            with urllib.request.urlopen(req) as resp: chunk = resp.read()\n            self.pos += len(chunk)\n            return chunk\n\n    rf = RangeFile(parquet_url)\n    pf = pq.ParquetFile(rf)\n    # Read specific row group or slice\n    # In CORD validation parquet, there is 1 row group with 100 rows\n    table = pf.read(columns=[\"bytes\"])\n    raw_png_bytes = table[\"bytes\"][row_idx].as_py()\n    rf.close()\n\n    img = Image.open(io.BytesIO(raw_png_bytes)).convert(\"RGB\")\n    return img\n\n\ndef run_feasibility_pipeline(\n    plan_path: Path,\n    output_dir: Path,\n    dry_run: bool = False,\n    device: str = \"cuda\",\n) -> Dict[str, Any]:\n    \"\"\"Execute pre-registered feasibility plan.\"\"\"\n    output_dir.mkdir(parents=True, exist_ok=True)\n    with open(plan_path, \"r\", encoding=\"utf-8\") as f:\n        plan = json.load(f)\n\n    meta = plan[\"metadata\"]\n    attempts = plan[\"attempts\"]\n    print(f\"=== Starting Document Generation Feasibility Study ===\")\n    print(f\"Study: {meta['study_name']} (Protocol v{meta['protocol_version']})\")\n    print(f\"Generator: {meta['model']['repository']} (Rev: {meta['model']['revision'][:8]})\")\n    print(f\"Target Output Directory: {output_dir.resolve()}\")\n\n    # 1. Hardware Preflight Check\n    preflight: Dict[str, Any] = {\n        \"timestamp\": time.strftime(\"%Y-%m-%dT%H:%M:%SZ\", time.gmtime()),\n        \"python_version\": sys.version,\n        \"platform\": sys.platform,\n        \"device_requested\": device,\n    }\n\n    disk_stat = shutil.disk_usage(output_dir)\n    preflight[\"disk_free_gb\"] = disk_stat.free / (1024**3)\n    print(f\"Disk free space: {preflight['disk_free_gb']:.2f} GB\")\n\n    has_cuda = False\n    try:\n        import torch\n        preflight[\"torch_version\"] = torch.__version__\n        has_cuda = torch.cuda.is_available()\n        preflight[\"cuda_available\"] = has_cuda\n        if has_cuda:\n            preflight[\"cuda_device_name\"] = torch.cuda.get_device_name(0)\n            preflight[\"cuda_device_count\"] = torch.cuda.device_count()\n            preflight[\"total_vram_gb\"] = torch.cuda.get_device_properties(0).total_memory / (1024**3)\n            print(f\"GPU: {preflight['cuda_device_name']} ({preflight['total_vram_gb']:.2f} GB VRAM)\")\n        else:\n            print(\"WARNING: CUDA is not available! Running on CPU.\")\n    except ImportError:\n        preflight[\"torch_version\"] = \"NOT_INSTALLED\"\n        preflight[\"cuda_available\"] = False\n        print(\"WARNING: PyTorch not installed.\")\n\n    preflight_path = output_dir / \"preflight_receipt.json\"\n    with open(preflight_path, \"w\", encoding=\"utf-8\") as f:\n        json.dump(preflight, f, indent=2)\n\n    if dry_run:\n        print(\"[DRY-RUN] Preflight completed successfully. Skipping model inference.\")\n        return {\"status\": \"DRY_RUN_COMPLETED\", \"preflight\": preflight}\n\n    if not has_cuda and device == \"cuda\":\n        raise RuntimeError(\"CUDA requested but not available. Aborting GPU execution.\")\n\n    # 2. Model Loading (SDXL Inpainting FP16 Safetensors)\n    from diffusers import StableDiffusionXLInpaintPipeline\n    import torch\n\n    repo_id = meta[\"model\"][\"repository\"]\n    revision = meta[\"model\"][\"revision\"]\n\n    print(f\"\\n[Stage 1] Loading SDXL Inpainting Pipeline ({repo_id} @ {revision[:8]})...\")\n    load_start = time.time()\n    pipe = StableDiffusionXLInpaintPipeline.from_pretrained(\n        repo_id,\n        revision=revision,\n        torch_dtype=torch.float16,\n        variant=\"fp16\",\n        use_safetensors=True,\n    )\n    pipe.to(\"cuda\")\n    # Enable VRAM memory optimizations for T4 compatibility\n    pipe.enable_attention_slicing()\n    load_duration = time.time() - load_start\n    print(f\"Model successfully loaded in {load_duration:.2f}s.\")\n\n    # Parquet URL\n    parquet_url = (\n        f\"https://huggingface.co/datasets/{meta['dataset']['repository']}/resolve/\"\n        f\"{meta['dataset']['revision']}/{meta['dataset']['parquet_file']}\"\n    )\n\n    # 3. Execute Attempts\n    attempt_results: List[Dict[str, Any]] = []\n    csv_rows: List[str] = [\n        \"attempt_id,source_image_id,operation,original_text,target_text,outside_bitwise_equal,inside_mean_l1,changed_pixel_ratio,preliminary_assessment,human_verdict,human_notes\"\n    ]\n\n    for attempt in attempts:\n        att_id = attempt[\"attempt_id\"]\n        att_dir = output_dir / att_id\n        att_dir.mkdir(parents=True, exist_ok=True)\n        print(f\"\\n--- Running Attempt: {att_id} ({attempt['operation'].upper()}) ---\")\n\n        # Resume check\n        receipt_path = att_dir / \"attempt_receipt.json\"\n        if receipt_path.exists():\n            print(f\"Receipt already exists for {att_id}. Skipping execution (resumed).\")\n            with open(receipt_path, \"r\", encoding=\"utf-8\") as rf:\n                att_res = json.load(rf)\n            attempt_results.append(att_res)\n            continue\n\n        # A. Fetch authentic source\n        print(f\"Fetching CORD-v2 source row {attempt['source_row_index']}...\")\n        auth_img = fetch_cord_source_image(attempt[\"source_row_index\"], parquet_url)\n        auth_path = att_dir / \"authentic_control.png\"\n        auth_img.save(auth_path, format=\"PNG\")\n\n        # B. Geometry ROI & Canvas Build\n        canvas_dict = extract_roi_and_build_canvas(\n            image=auth_img,\n            target_bbox=attempt[\"target_bbox\"],\n            canvas_size=meta[\"model\"][\"canvas_size\"],\n            context_margin_ratio=attempt[\"context_margin_ratio\"],\n        )\n        context_path = att_dir / \"generation_context.png\"\n        canvas_dict[\"context_crop\"].save(context_path, format=\"PNG\")\n\n        geo_receipt_path = att_dir / \"geometry_receipt.json\"\n        with open(geo_receipt_path, \"w\", encoding=\"utf-8\") as gf:\n            json.dump(canvas_dict[\"geometry_receipt\"], gf, indent=2)\n\n        # C. Inference\n        print(f\"Executing SDXL inpainting (Seed={attempt['seed']}, Steps={meta['model']['num_inference_steps']})...\")\n        generator = torch.Generator(device=\"cuda\").manual_seed(attempt[\"seed\"])\n        inf_start = time.time()\n        with torch.inference_mode():\n            out = pipe(\n                prompt=attempt[\"prompt\"],\n                negative_prompt=attempt[\"negative_prompt\"],\n                image=canvas_dict[\"canvas_image\"],\n                mask_image=canvas_dict[\"canvas_mask\"],\n                num_inference_steps=meta[\"model\"][\"num_inference_steps\"],\n                guidance_scale=meta[\"model\"][\"guidance_scale\"],\n                strength=meta[\"model\"][\"strength\"],\n                generator=generator,\n            )\n        inf_duration = time.time() - inf_start\n        raw_output_img = out.images[0]\n        raw_output_path = att_dir / \"raw_model_output.png\"\n        raw_output_img.save(raw_output_path, format=\"PNG\")\n\n        # D. Inverse Mapping & Composite\n        print(\"Compositing into original document coordinates...\")\n        comp_res = composite_inpainted_patch(\n            original_image=auth_img,\n            model_output_canvas=raw_output_img,\n            geometry_receipt=canvas_dict[\"geometry_receipt\"],\n        )\n        final_edited = comp_res[\"final_edited_image\"]\n        final_mask = comp_res[\"final_intervention_mask\"]\n\n        edited_path = att_dir / \"final_edited_document.png\"\n        mask_path = att_dir / \"final_intervention_mask.png\"\n        final_edited.save(edited_path, format=\"PNG\")\n        final_mask.save(mask_path, format=\"PNG\")\n\n        # E. Technical Verification\n        is_outside_equal, max_outside_diff = verify_outside_equality(auth_img, final_edited, final_mask)\n        print(f\"Outside-mask bitwise equal: {is_outside_equal} (Max diff: {max_outside_diff})\")\n\n        # Pixel change statistics\n        auth_arr = np.array(auth_img, dtype=np.int32)\n        edit_arr = np.array(final_edited, dtype=np.int32)\n        mask_arr = np.array(final_mask, dtype=np.uint8) == 255\n\n        in_diff = np.abs(auth_arr[mask_arr] - edit_arr[mask_arr])\n        in_mean_l1 = float(np.mean(in_diff)) if np.any(mask_arr) else 0.0\n        changed_count = int(np.sum(np.any(in_diff > 0, axis=-1)))\n        total_in = int(np.sum(mask_arr))\n        changed_ratio = float(changed_count / total_in) if total_in > 0 else 0.0\n\n        print(f\"Inside-mask mean L1: {in_mean_l1:.2f}, Changed pixel ratio: {changed_ratio:.2%}\")\n\n        # Preliminary Assessment (Heuristic description)\n        preliminary = \"TECHNICAL_VALID\"\n        if not is_outside_equal:\n            preliminary = \"TECHNICAL_FAIL_OUTSIDE_LEAKAGE\"\n        elif in_mean_l1 < 1.0 or changed_ratio < 0.1:\n            preliminary = \"POTENTIAL_UNMODIFIED\"\n        else:\n            preliminary = f\"FEASIBLE_{attempt['operation'].upper()}\"\n\n        receipt_data = {\n            \"attempt_id\": att_id,\n            \"source_doc_identifier\": attempt[\"source_doc_identifier\"],\n            \"operation\": attempt[\"operation\"],\n            \"original_text\": attempt[\"original_text\"],\n            \"target_text\": attempt[\"target_text\"],\n            \"inference_duration_sec\": inf_duration,\n            \"peak_vram_mb\": torch.cuda.max_memory_allocated() / (1024**2) if has_cuda else 0.0,\n            \"technical_qc\": {\n                \"is_outside_bitwise_equal\": is_outside_equal,\n                \"max_outside_diff\": max_outside_diff,\n                \"inside_mean_l1\": in_mean_l1,\n                \"changed_pixel_count\": changed_count,\n                \"changed_pixel_ratio\": changed_ratio,\n            },\n            \"file_hashes\": {\n                \"authentic_control_sha256\": compute_sha256(auth_path),\n                \"generation_context_sha256\": compute_sha256(context_path),\n                \"raw_model_output_sha256\": compute_sha256(raw_output_path),\n                \"final_edited_document_sha256\": compute_sha256(edited_path),\n                \"final_intervention_mask_sha256\": compute_sha256(mask_path),\n            },\n            \"preliminary_assessment\": preliminary,\n        }\n\n        with open(receipt_path, \"w\", encoding=\"utf-8\") as rf:\n            json.dump(receipt_data, rf, indent=2)\n        attempt_results.append(receipt_data)\n\n        csv_rows.append(\n            f\"{att_id},{attempt['source_image_id']},{attempt['operation']},\"\n            f\"\\\"{attempt['original_text']}\\\",\\\"{attempt['target_text']}\\\",\"\n            f\"{is_outside_equal},{in_mean_l1:.2f},{changed_ratio:.4f},{preliminary},\"\n            f\"PENDING_HUMAN_REVIEW,\\\"\\\")\"\n        )\n\n    # 4. Write Human Review CSV\n    csv_path = output_dir / \"human_review.csv\"\n    with open(csv_path, \"w\", encoding=\"utf-8\") as cf:\n        cf.write(\"\\n\".join(csv_rows) + \"\\n\")\n    print(f\"\\nWritten {csv_path}\")\n\n    # 5. Build Final Manifest & Zip Archive\n    manifest = {\n        \"metadata\": meta,\n        \"preflight\": preflight,\n        \"model_load_duration_sec\": load_duration,\n        \"attempts_summary\": attempt_results,\n    }\n    manifest_path = output_dir / \"feasibility_manifest.json\"\n    with open(manifest_path, \"w\", encoding=\"utf-8\") as mf:\n        json.dump(manifest, mf, indent=2)\n\n    zip_path = output_dir / \"document_feasibility_results.zip\"\n    print(f\"Packaging archive to {zip_path}...\")\n    with zipfile.ZipFile(zip_path, \"w\", zipfile.ZIP_DEFLATED) as zf:\n        for root, _, files in os.walk(output_dir):\n            for file in files:\n                p = Path(root) / file\n                if p == zip_path:\n                    continue\n                rel_p = p.relative_to(output_dir)\n                zf.write(p, rel_p)\n\n    archive_size_mb = zip_path.stat().st_size / (1024**2)\n    print(f\"Archive created successfully: {zip_path} ({archive_size_mb:.2f} MB)\")\n    print(\"=== Generation Feasibility Study Execution Complete ===\")\n\n    return {\n        \"status\": \"EXECUTED_PENDING_HUMAN_QC\",\n        \"archive_path\": str(zip_path),\n        \"archive_size_mb\": archive_size_mb,\n        \"attempts\": attempt_results,\n    }\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=\"Run Document Feasibility Study on Colab GPU.\")\n    parser.add_argument(\"--plan\", type=Path, default=Path(\"research/evidence/document_pilot/document_feasibility_plan.json\"))\n    parser.add_argument(\"--output-dir\", type=Path, default=Path(\"research/evidence/document_pilot/feasibility_results\"))\n    parser.add_argument(\"--dry-run\", action=\"store_true\", help=\"Perform preflight checks without downloading model or inferencing.\")\n    parser.add_argument(\"--device\", type=str, default=\"cuda\", choices=[\"cuda\", \"cpu\"])\n    args = parser.parse_args()\n\n    run_feasibility_pipeline(\n        plan_path=args.plan,\n        output_dir=args.output_dir,\n        dry_run=args.dry_run,\n        device=args.device,\n    )\n\n\nif __name__ == \"__main__\":\n    main()\n")
print("✓ Đã ghi runner script vào: run_document_feasibility_colab.py")


In [ ]:
# Cell 3: Thực thi chạy 4 attempts tạo dữ liệu khả thi
!python run_document_feasibility_colab.py --plan document_feasibility_plan.json --output-dir ./results


In [ ]:
# Cell 4: Kiểm tra kết quả và tải file archive về LOCAL
from pathlib import Path
from google.colab import files

zip_path = Path('./results/document_feasibility_results.zip')
if zip_path.exists():
    size_mb = zip_path.stat().st_size / (1024**2)
    print(f"✓ Archive hoàn tất: {zip_path.name} ({size_mb:.2f} MB)")
    print("Đang tải archive về máy LOCAL...")
    files.download(str(zip_path))
    print("\nSau khi tải xong, hãy đặt file vào thư mục dự án và chạy:")
    print("  python scripts/research/verify_document_feasibility_artifacts.py --zip document_feasibility_results.zip")
else:
    print("LỖI: Không tìm thấy file archive kết quả! Hãy kiểm tra log Cell 3.")
